In [0]:
from pyspark.sql.functions import *
# Get batch_id from widget
dbutils.widgets.text("batch_id", "")
batch_id = dbutils.widgets.get("batch_id")
print(batch_id)

100


In [0]:
# Read metadata for this batch
master_tbl_df = spark.sql(f"SELECT * FROM metadata.master_tble WHERE batch_id = '{batch_id}'")
display(master_tbl_df)

batch_id,source_name,data_load_strategy,source_tbl_name,bronze_tbl_name,silver_tbl_name,watermark_column
100,azure_sql,FullLoad,dbo.product,bronze.product,silver.product,2000-09-07T15:23:08.535Z


In [0]:
#extract values onece from the master_tbl_df
row=master_tbl_df.first()
src_tbl_name_dynamic=row["source_tbl_name"]
bronze_tbl_name_dynamic=row["bronze_tbl_name"]
silver_tbl_name_dynamic=row["silver_tbl_name"]
data_load_strategy_dynamic=row["data_load_strategy"]
prev_modified_date_dynamic=row["watermark_column"]

print("Source:",src_tbl_name_dynamic)
print("Bronze:",bronze_tbl_name_dynamic)
print("Silver:",silver_tbl_name_dynamic)
print("Strategy:",data_load_strategy_dynamic)
print("Prev Watermark:",prev_modified_date_dynamic)

Source: dbo.product
Bronze: bronze.product
Silver: silver.product
Strategy: FullLoad
Prev Watermark: 2000-09-07 15:23:08.535000


In [0]:
src_url=(
    f"jdbc:sqlserver://dm-project1.database.windows.net:1433;"
    f"database=Sales_db;"
    f"user=dmuser;"
    f"password=Dipti123@+;"
)

In [0]:
if data_load_strategy_dynamic=="FullLoad":
    #Read Source
    source_df=spark.read\
        .format('jdbc')\
        .option("url",src_url)\
        .option("dbtable",src_tbl_name_dynamic)\
        .load()
    display(source_df)

    #Compute max ModifiedDate
    src_max_modified_date=source_df.agg(max("ModifiedDate")).first()[0]
    print("New Watermark:",src_max_modified_date)

    #Add audit column
    source_df_add_col=source_df.withColumn("load_date",current_timestamp())

    #Write to Bronze and Silver directly
    source_df_add_col.write.mode("overwrite").saveAsTable(bronze_tbl_name_dynamic)
    source_df_add_col.write.mode("overwrite").saveAsTable(silver_tbl_name_dynamic)

    #update metadata watermark
    spark.sql("""
              UPDATE metadata.master_tble
              SET watermark_column='{src_max_modified_date}'
              WHERE batch_id='{batch_id}'
            """)
else:
    print("Invalid batch_id or unsupported starategy")


ProductID,Name,StandardCost,ModifiedDate
501,Laptop Pro 15,1200.00,2025-11-05T00:00:00.000Z
502,Smartphone X,800.00,2025-11-06T00:00:00.000Z
503,Wireless Headphones,150.00,2025-11-07T00:00:00.000Z
504,null,500.00,null
505,Tablet Z,-300.00,2025-11-08T00:00:00.000Z
506,Laptop Pro 15,1200.00,2025-11-09T00:00:00.000Z
507,Gaming Console Y,400.00,2025-11-10T00:00:00.000Z
508,Smartwatch S,250.00,2025-11-11T00:00:00.000Z
509,Monitor UltraWide,350.00,2025-11-12T00:00:00.000Z
510,Keyboard Mechanical,90.00,2025-11-13T00:00:00.000Z


New Watermark: 2025-11-13 00:00:00


---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-8022700542582240>, line 22
     19     source_df_add_col.write.mode("overwrite").saveAsTable(silver_tbl_name_dynamic)
     21     #update metadata watermark
---> 22     spark.sql("""
     23               UPDATE metadata.master_tble
     24               SET watermark_column='{src_max_modified_date}'
     25               WHERE batch_id='{batch_id}'
     26             """)
     27 else:
     28     print("Invalid batch_id or unsupported starategy")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:875, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    872         _views.append(SubqueryAlias(df._plan, name))
    874 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 875 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    876 if "sql_command_result" in 

In [0]:
%sql
select * from bronze.product;

ProductID,Name,StandardCost,ModifiedDate,load_date
501,Laptop Pro 15,1200.00,2025-11-05T00:00:00.000Z,2025-11-30T08:41:36.592Z
502,Smartphone X,800.00,2025-11-06T00:00:00.000Z,2025-11-30T08:41:36.592Z
503,Wireless Headphones,150.00,2025-11-07T00:00:00.000Z,2025-11-30T08:41:36.592Z
504,null,500.00,null,2025-11-30T08:41:36.592Z
505,Tablet Z,-300.00,2025-11-08T00:00:00.000Z,2025-11-30T08:41:36.592Z
506,Laptop Pro 15,1200.00,2025-11-09T00:00:00.000Z,2025-11-30T08:41:36.592Z
507,Gaming Console Y,400.00,2025-11-10T00:00:00.000Z,2025-11-30T08:41:36.592Z
508,Smartwatch S,250.00,2025-11-11T00:00:00.000Z,2025-11-30T08:41:36.592Z
509,Monitor UltraWide,350.00,2025-11-12T00:00:00.000Z,2025-11-30T08:41:36.592Z
510,Keyboard Mechanical,90.00,2025-11-13T00:00:00.000Z,2025-11-30T08:41:36.592Z


In [0]:
%sql
SELECT * FROM silver.product;

ProductID,Name,StandardCost,ModifiedDate,load_date
501,Laptop Pro 15,1200.00,2025-11-05T00:00:00.000Z,2025-11-30T08:41:38.053Z
502,Smartphone X,800.00,2025-11-06T00:00:00.000Z,2025-11-30T08:41:38.053Z
503,Wireless Headphones,150.00,2025-11-07T00:00:00.000Z,2025-11-30T08:41:38.053Z
504,null,500.00,null,2025-11-30T08:41:38.053Z
505,Tablet Z,-300.00,2025-11-08T00:00:00.000Z,2025-11-30T08:41:38.053Z
506,Laptop Pro 15,1200.00,2025-11-09T00:00:00.000Z,2025-11-30T08:41:38.053Z
507,Gaming Console Y,400.00,2025-11-10T00:00:00.000Z,2025-11-30T08:41:38.053Z
508,Smartwatch S,250.00,2025-11-11T00:00:00.000Z,2025-11-30T08:41:38.053Z
509,Monitor UltraWide,350.00,2025-11-12T00:00:00.000Z,2025-11-30T08:41:38.053Z
510,Keyboard Mechanical,90.00,2025-11-13T00:00:00.000Z,2025-11-30T08:41:38.053Z
